<a href="https://colab.research.google.com/github/praksb2428-maker/m_l_practice/blob/main/09_agentic_recommender_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 에이전트 AI 기반 추천 시스템

## 1. 최신 추천 시스템 동향 소개
- 기존 추천 시스템의 한계
  - 협업 필터링(CF) 및 행렬 분해(MF) 등 전통적 기법은 행렬 형태의 평점 데이터나 클릭 로그에만 의존
  - 사용자의 복잡한 자연어 맥락이나 실시간으로 변하는 의도를 정밀하게 반영하기 어려움
  
  
- 에이전트 AI기반 추천 시스템 등장
  - 최근 추천 시스템의 SOTA는 LLM과 자율 에이전트 워크플로우를 접목하는 방향으로 진화중
  - 에이전트가 사용자의 모호한 요청을 받으면 스스로 계획을 세우고, 필요한 도구를 호출하여 최적 결과 도출

## 2. 환경 설정 및 데이터베이스 구축
- 아이템 데이터베이스 정의
  - 에이전트가 탐색할 영화 데이터베이스 구축
  - 단순한 장르 표기를 넘어 에이전트가 텍스트 분석을 통해 판단할 수 있도록 상세 리뷰 및 키워드를 포함함
  
- **에이전트가 사용할 도구(Tools) 구현**
  - `search_by_genre`: 특정 장르의 영화를 필터링하는 도구
  - `analyze_reviews`: 사용자의 감성이나 특정 분위기 요구사항이 리뷰 데이터에 포함되어 있는지 검증하는 도구

In [4]:
import pandas as pd
import numpy as np

# 1. Define sample movie database (Item Pool)
movie_db = [
    {"title": "La La Land", "genre": "Romance", "reviews": "Beautiful music, romantic but slightly bittersweet ending, perfect for quiet night.", "rating": 4.5},
    {"title": "About Time", "genre": "Romance", "reviews": "Warm and emotional time-travel story about family and true love.", "rating": 4.7},
    {"title": "Interstellar", "genre": "Sci-Fi", "reviews": "Mind-bending space exploration, cosmic mystery, deeply emotional father-daughter bond.", "rating": 4.8},
    {"title": "Inception", "genre": "Sci-Fi", "reviews": "Intense action, dream architecture, complex plot, highly intellectual thriller.", "rating": 4.6},
    {"title": "The Conjuring", "genre": "Horror", "reviews": "Terrifying atmosphere, jump scares, classic haunted house mystery.", "rating": 4.2},
    {"title": "Get Out", "genre": "Horror", "reviews": "Social satire combined with psychological tension, mysterious and suspenseful.", "rating": 4.4}
]
df_movies = pd.DataFrame(movie_db)

# 2. Tool 1: Genre filtering tool 정확히 일치하는 목록만을 추출하는 도구 함수
def search_by_genre(genre_query):
    """Filters movies that exactly match the requested genre."""
    return df_movies[df_movies['genre'].str.lower() == genre_query.lower()]

# 3. Tool 2: Review keyword analysis tool 원하는 키워드만 필터링하는 도구 함수
def analyze_reviews(sub_df, keyword):
    """Searches for specific semantic keywords within movie reviews."""
    return sub_df[sub_df['reviews'].str.lower().str.contains(keyword.lower())]

print("Tools and Database initialized successfully!")

Tools and Database initialized successfully!


## 3. 추천 에이전트  로직 구현
- **에이전트의 3단계 워크플로우**
  - 1단계 Planning : 사용자의 모호한 자연어 요청에서 핵심 장르와 원하는 '분위기 키워드'를 스스로 분리 추출
  - 2단계 Tool Calling : 1단계에서 세운 계획에 따라 `search_by_genre`와 `analyze_reviews` 도구를 순차적으로 호출
  - 3단계 최종 추천 사유 생성 : 필터링된 결과 중 평점이 가장 높은 최적의 아이템을 선정하고 추천 이유를 자연어 형태로 출력

In [5]:
# 자율형 ai 에이전트 엔진코드
class RecommenderAgent:
    def __init__(self, database):
        self.db = database

    def run(self, user_request):
        print(f"[User Request]: '{user_request}'\n")

        # --- Step 1: Planning & Parsing (Simulating LLM Understanding) ---
        # Agent autonomously extracts structural requirements from unstructured text
        extracted_genre = None
        extracted_keyword = None
        # extracted_genre 정의
        if "우주" in user_request or "sf" in user_request.lower():
            extracted_genre = "Sci-Fi"
        elif "로맨스" in user_request or "사랑" in user_request:
            extracted_genre = "Romance"
        elif "무서운" in user_request or "공포" in user_request:
            extracted_genre = "Horror"
        # extracted_keyword 정의
        if "감동" in user_request or "family" in user_request or "가족" in user_request:
            extracted_keyword = "family"
        elif "지적" in user_request or "어려운" in user_request or "complex" in user_request:
            extracted_keyword = "complex"
        elif "잔잔한" in user_request or "quiet" in user_request:
            extracted_keyword = "quiet"

        print(f"[Step 1: Planning] Agent parsed parameters -> Genre: {extracted_genre}, Vibe: {extracted_keyword}")

        # --- Step 2: Tool Calling ---
        # Call Tool 1: Initial filtering
        if extracted_genre:
            candidates = search_by_genre(extracted_genre)
            print(f"[Step 2-1: Tool Call] Filtered by genre. Found {len(candidates)} candidates.")
        else:
            candidates = self.db

        # Call Tool 2: Deep review contextual analysis
        if extracted_keyword and not candidates.empty:
            candidates = analyze_reviews(candidates, extracted_keyword)
            print(f"[Step 2-2: Tool Call] Filtered by review sentiment/vibe. Found {len(candidates)} candidates.")

        # --- Step 3: Final Selection & Reasoning ---
        if candidates.empty:
            return "[Agent Output]: 죄송합니다. 요청하신 조건에 맞는 최적의 추천 아이템을 찾지 못했습니다."

        # Select the best item based on rating among qualified candidates
        best_item = candidates.sort_values(by='rating', ascending=False).iloc[0]

        output_message = (
            f"[Agent Output: Recommendation Result]\n"
            f"- 추천 시스템이 선정한 최적의 아이템: {best_item['title']}\n"
            f"- 선정 이유: 이 영화는 고객님이 원하신 {extracted_genre} 장르에 속하며, "
            f"리뷰 분석 결과 '{best_item['reviews']}'라는 특징을 가지고 있어 요구사항에 가장 부합 (평점: {best_item['rating']})"
        )
        return output_message

# Instantiate the agent
agent = RecommenderAgent(df_movies)

## 4. 에이전트 기반 추천 실행 및 결과 확인
- **테스트 시나리오 수행**
  - 사용자가 자연어로 "가족들과 함께 볼 수 있는 감동적인 로맨스 영화를 골라줘"라고 복잡한 맥락의 요청을 보냄
  - 에이전트가 중간에 도구들을 적절히 조합하여 최종 결론과 인간이 이해할 수 있는 추천 사유를 도출하는지 확인

In [6]:
# Run the system with a complex, natural language request
user_query = "가족들과 함께 볼 수 있는 따뜻하고 감동적인 로맨스 영화를 골라줘"
response = agent.run(user_query)
print("\n" + response)

[User Request]: '가족들과 함께 볼 수 있는 따뜻하고 감동적인 로맨스 영화를 골라줘'

[Step 1: Planning] Agent parsed parameters -> Genre: Romance, Vibe: family
[Step 2-1: Tool Call] Filtered by genre. Found 2 candidates.
[Step 2-2: Tool Call] Filtered by review sentiment/vibe. Found 1 candidates.

[Agent Output: Recommendation Result]
- 추천 시스템이 선정한 최적의 아이템: About Time
- 선정 이유: 이 영화는 고객님이 원하신 Romance 장르에 속하며, 리뷰 분석 결과 'Warm and emotional time-travel story about family and true love.'라는 특징을 가지고 있어 요구사항에 가장 부합 (평점: 4.7)


## 5. 확인 문제

문제 1. 전통적인 추천 기법과 비교했을 때, 본 실습의 에이전트 기반 추천 시스템이 가지는 핵심적인 장점과 차이점은 무엇인지 적기

anser : 장르를 필터링하여 선정하고 영화의 리뷰 텍스트를 분석하여 선호하는것을 추천함


문제 2. 코드 빈칸 채우기
- 사용자가 "지적이고 구조가 복잡한 SF 영화를 추천해줘"라고 요청했을 때, 위 에이전트가 코드 내부적으로 `extracted_genre`와 `extracted_keyword`를 각각 어떤 값으로 추출하게 될까?
  - `extracted_genre` = [ Sci-Fi  ]
  - `extracted_keyword` = [ complex ]